<a href="https://colab.research.google.com/github/nurfnick/Numerical_Methods/blob/master/Module_6_Regression/LinearRegressionVibeCoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Generate Synthetic Student Attendance Dataset
We will create a dataset with 12 unique students, each having multiple historical entries. Factors included:
- `sessions_attended`: Number of previous sessions attended.
- `letter_grade`: Encoded as GPA equivalent (A=4, B=3, C=2, D=1, F=0).
- `year_in_school`: 1 (Freshman) to 4 (Senior).
- `is_festival_day`: Binary flag (1 if there is a campus festival, 0 otherwise).
- `homecoming_commitments`: Binary flag (1 if involved in homecoming, 0 otherwise).
- `weather_index`: Score from 1 (terrible weather) to 5 (excellent weather).
- `attended`: Target variable (1 if attended, 0 if absent).

In [1]:
import numpy as np
import pandas as pd

# Set seed for reproducibility
np.random.seed(42)

num_students = 12
records_per_student = 15
total_records = num_students * records_per_student

student_ids = [f"Student_{i+1:02d}" for i in range(num_students)]

data = {
    'student_id': np.repeat(student_ids, records_per_student),
    'sessions_attended': np.random.randint(0, 20, size=total_records),
    'current_grade_val': np.random.choice([4.0, 3.0, 2.0, 1.0, 0.0], size=total_records, p=[0.3, 0.4, 0.2, 0.08, 0.02]),
    'year_in_school': np.repeat(np.random.randint(1, 5, size=num_students), records_per_student),
    'is_festival_day': np.random.choice([0, 1], size=total_records, p=[0.85, 0.15]),
    'homecoming_commitments': np.random.choice([0, 1], size=total_records, p=[0.9, 0.1]),
    'weather_index': np.random.randint(1, 6, size=total_records)
}

df = pd.DataFrame(data)

# Generate log odds to determine the probability of attendance
# Higher sessions attended, higher grade, better weather increase attendance chance
# Festivals, homecoming commitments decrease attendance chance
log_odds = (
    0.15 * df['sessions_attended'] +
    0.5 * df['current_grade_val'] -
    0.2 * df['year_in_school'] -
    1.5 * df['is_festival_day'] -
    1.2 * df['homecoming_commitments'] +
    0.3 * df['weather_index'] -
    1.5
)

prob = 1 / (1 + np.exp(-log_odds))
df['attended'] = np.random.binomial(1, prob)

print("Dataset shape:", df.shape)
display(df.head(10))

Dataset shape: (180, 8)


,student_id,sessions_attended,current_grade_val,year_in_school,is_festival_day,homecoming_commitments,weather_index,attended
0,Student_01,6,3.0,4,0,0,5,1
1,Student_01,19,3.0,4,0,0,3,0
2,Student_01,14,3.0,4,0,0,3,1
3,Student_01,10,4.0,4,0,0,1,1
4,Student_01,7,3.0,4,0,0,2,1
5,Student_01,6,4.0,4,0,0,2,0
6,Student_01,18,2.0,4,1,0,5,1
7,Student_01,10,3.0,4,1,0,2,1
8,Student_01,10,0.0,4,0,0,4,1
9,Student_01,3,3.0,4,0,0,2,0


### 2. Train the Linear Model (Logistic Regression)
We will train a logistic regression model on this dataset to predict the probability of a student attending class.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

features = ['sessions_attended', 'current_grade_val', 'year_in_school', 'is_festival_day', 'homecoming_commitments', 'weather_index']
X = df[features]
y = df['attended']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print(classification_report(y_test, y_pred))

Model Accuracy: 0.81
              precision    recall  f1-score   support

           0       1.00      0.30      0.46        10
           1       0.79      1.00      0.88        26

    accuracy                           0.81        36
   macro avg       0.89      0.65      0.67        36
weighted avg       0.85      0.81      0.76        36



### 3. Predict Tuesday's Attendance for All 12 Students
We will define the conditions for next Tuesday:
- High weather quality index (e.g., 4 out of 5).
- No festival day (0).
- No homecoming commitments (0).
- We will use each student's latest records for `sessions_attended`, `current_grade_val`, and `year_in_school` to make the prediction.

In [3]:
# Get the latest profile status for each of the 12 students
latest_student_profiles = df.groupby('student_id').last().reset_index()

# Set Tuesday's conditions
latest_student_profiles['is_festival_day'] = 0
latest_student_profiles['homecoming_commitments'] = 0
latest_student_profiles['weather_index'] = 4  # e.g., Nice autumn day

# Predict
predictions_prob = model.predict_proba(latest_student_profiles[features])[:, 1]
predictions_class = model.predict(latest_student_profiles[features])

results = pd.DataFrame({
    'Student ID': latest_student_profiles['student_id'],
    'Sessions Attended': latest_student_profiles['sessions_attended'],
    'Current Grade': latest_student_profiles['current_grade_val'],
    'Year in School': latest_student_profiles['year_in_school'],
    'Attendance Probability': predictions_prob,
    'Will Show Up': predictions_class
})

display(results.sort_values(by='Attendance Probability', ascending=False))

,Student ID,Sessions Attended,Current Grade,Year in School,Attendance Probability,Will Show Up
1,Student_02,18,3.0,1,0.979531,1
7,Student_08,18,3.0,4,0.966210,1
4,Student_05,12,3.0,2,0.941482,1
11,Student_12,11,3.0,2,0.932456,1
2,Student_03,14,2.0,3,0.924555,1
5,Student_06,9,2.0,2,0.871269,1
9,Student_10,5,3.0,4,0.796302,1
0,Student_01,5,3.0,4,0.796302,1
3,Student_04,9,0.0,1,0.780829,1
8,Student_09,9,0.0,1,0.780829,1
